In [41]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

data_list = [(100, "Prashant", 45, 45000.00),
             (101, "Tarun", 36, 33000.00),
             (102, "David", 48, 28000.00)]

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("age", IntegerType()),
    StructField("salary", FloatType())
])

# df = spark.createDataFrame(data_list).toDF("id", "name", "age", "salary")
df = spark.createDataFrame(data_list, schema)
df.show()


+---+--------+---+-------+
| id|    name|age| salary|
+---+--------+---+-------+
|100|Prashant| 45|45000.0|
|101|   Tarun| 36|33000.0|
|102|   David| 48|28000.0|
+---+--------+---+-------+



In [27]:
# Add increment column that is 10% of salary upto 3000 and revise salary that is old + increment
from pyspark.sql.functions import expr

# using chain of withColumn in order to create increment first and the revised salary
# revised_df = df.withColumn("increment", expr("case when salary > 30000 then 3000 else (0.1 * salary) end")).withColumn("revised_salary", expr("increment + salary"))

# using withColumns to create both columns at once
revised_df = df.withColumns({
    "increment": expr("case when salary > 30000 then 3000 else (0.1 * salary) end"),
    "revised_salary": expr("case when salary > 30000 then 3000 else (0.1 * salary) end + salary")
})

revised_df.show()

+---+--------+---+-------+---------+--------------+
| id|    name|age| salary|increment|revised_salary|
+---+--------+---+-------+---------+--------------+
|100|Prashant| 45|45000.0|   3000.0|       48000.0|
|101|   Tarun| 36|33000.0|   3000.0|       36000.0|
|102|   David| 48|28000.0|   2800.0|       30800.0|
+---+--------+---+-------+---------+--------------+



In [34]:
# add a batch number UUID column
import uuid
from pyspark.sql.functions import lit

batch_id = str(uuid.uuid4())

revised_batch_df = revised_df.withColumn("batch_id", lit(batch_id))

revised_batch_df.show()

+---+--------+---+-------+---------+--------------+--------------------+
| id|    name|age| salary|increment|revised_salary|            batch_id|
+---+--------+---+-------+---------+--------------+--------------------+
|100|Prashant| 45|45000.0|   3000.0|       48000.0|3800c98f-3bbc-4e3...|
|101|   Tarun| 36|33000.0|   3000.0|       36000.0|3800c98f-3bbc-4e3...|
|102|   David| 48|28000.0|   2800.0|       30800.0|3800c98f-3bbc-4e3...|
+---+--------+---+-------+---------+--------------+--------------------+



In [36]:
"""
Rename the dataframe colums as listed below
    increment - annual_increment
    salary - incremented_salary
"""

renamed_df = revised_batch_df.withColumnsRenamed({"increment": "annual_increment", "salary": "incremented_salary"})
renamed_df.show()

+---+--------+---+------------------+----------------+--------------+--------------------+
| id|    name|age|incremented_salary|annual_increment|revised_salary|            batch_id|
+---+--------+---+------------------+----------------+--------------+--------------------+
|100|Prashant| 45|           45000.0|          3000.0|       48000.0|3800c98f-3bbc-4e3...|
|101|   Tarun| 36|           33000.0|          3000.0|       36000.0|3800c98f-3bbc-4e3...|
|102|   David| 48|           28000.0|          2800.0|       30800.0|3800c98f-3bbc-4e3...|
+---+--------+---+------------------+----------------+--------------+--------------------+



In [40]:
"""
Remove the following colums from your dataframe
    age
    annual_increment
"""

dropped_df = renamed_df.drop("age", "annual_increment")
dropped_df.show()

+---+--------+------------------+--------------+--------------------+
| id|    name|incremented_salary|revised_salary|            batch_id|
+---+--------+------------------+--------------+--------------------+
|100|Prashant|           45000.0|       48000.0|3800c98f-3bbc-4e3...|
|101|   Tarun|           33000.0|       36000.0|3800c98f-3bbc-4e3...|
|102|   David|           28000.0|       30800.0|3800c98f-3bbc-4e3...|
+---+--------+------------------+--------------+--------------------+

